# Practice Lab: Math Foundations 2 - How Models Learn

8 exercises on cross-entropy, gradient descent, learning rate, backpropagation.

In [ ]:
!pip install -q numpy torch matplotlib scikit-learn

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
print('torch:', torch.__version__)

## Exercise 1 - Cross-entropy by hand

**Task:** Given logits and a true class index, compute the cross-entropy loss manually (stable softmax then negative log-likelihood) and verify it matches `F.cross_entropy`.

**Expected:** `ce_hand` and the PyTorch value agree within 1e-5.

In [ ]:
# YOUR CODE HERE
logits = np.array([1.5, 0.5, 2.0])
true_idx = 2

# TODO: compute cross-entropy by hand
#   1. shift logits by their max, exponentiate, normalize -> probs
#   2. ce_hand = -log(probs[true_idx])
# Then verify against F.cross_entropy and assert they match within 1e-5

<details><summary>💡 Hint</summary>

Stable softmax: `e = np.exp(logits - logits.max()); probs = e / e.sum()`. Cross-entropy for the true class is `-np.log(probs[true_idx])`.
</details>

**Criteria:** `ce_hand` printed and matches `F.cross_entropy` within 1e-5.

<details><summary>✅ Solution</summary>


In [ ]:
logits = np.array([1.5, 0.5, 2.0])
true_idx = 2

shifted = logits - logits.max()
exps = np.exp(shifted)
probs = exps / exps.sum()
ce_hand = -np.log(probs[true_idx])
print(f'by hand: {ce_hand:.6f}')

ce_pt = F.cross_entropy(torch.tensor(logits).unsqueeze(0).float(), torch.tensor([true_idx]))
print(f'pytorch: {ce_pt.item():.6f}')
assert abs(ce_hand - ce_pt.item()) < 1e-5

</details>

---

## Exercise 2 - Three LR regimes

**Task:** Minimize `(w - 3)^2` with plain gradient descent at three learning rates and label each run as too-slow, converged, or diverged.

**Expected:** LR=0.001 barely moves, LR=0.1 converges near w=3, LR=1.5 diverges.

In [ ]:
# YOUR CODE HERE
def loss(w): return (w - 3) ** 2
def grad(w): return 2 * (w - 3)

def descend(lr, steps=50, w0=0.0):
    w = w0
    # TODO: run `steps` updates of  w = w - lr * grad(w)
    #       if |w| blows past 1e10, bail out and return (inf, inf)
    ...
    return w, loss(w)

# TODO: for lr in [0.001, 0.1, 1.5], print w_final and final loss

<details><summary>💡 Hint</summary>

One update is `w = w - lr * grad(w)`. A tiny LR crawls, a mid LR converges, and a large LR (1.5 here) overshoots and diverges.
</details>

**Criteria:** All three LRs run; LR=0.1 converges near w=3, LR=1.5 diverges.

<details><summary>✅ Solution</summary>


In [ ]:
def loss(w): return (w - 3) ** 2
def grad(w): return 2 * (w - 3)

def descend(lr, steps=50, w0=0.0):
    w = w0
    for _ in range(steps):
        w = w - lr * grad(w)
        if abs(w) > 1e10: return float('inf'), float('inf')
    return w, loss(w)

for lr in [0.001, 0.1, 1.5]:
    w_final, l_final = descend(lr)
    print(f'LR={lr:.3f}  w_final={w_final:.3e}  loss={l_final:.3e}')

</details>

---

## Exercise 3 - Manual gradient vs autograd

**Task:** For `f = w0^2 + 2*w1^2`, derive the gradient by hand, then confirm it against PyTorch autograd.

**Expected:** `w.grad` equals `[2.0, 4.0]`.

In [ ]:
# YOUR CODE HERE
w = torch.tensor([1.0, 1.0], requires_grad=True)
loss = w[0] ** 2 + 2 * w[1] ** 2

# TODO: call loss.backward(), then print w.grad
#       Derive it by hand too:  d/dw0 = 2*w0,  d/dw1 = 4*w1  ->  [2.0, 4.0]

<details><summary>💡 Hint</summary>

For `f = w0^2 + 2*w1^2` the partials are `2*w0` and `4*w1`. `loss.backward()` fills `w.grad`.
</details>

**Criteria:** Autograd `w.grad` equals the hand-derived `[2.0, 4.0]`.

<details><summary>✅ Solution</summary>


In [ ]:
w = torch.tensor([1.0, 1.0], requires_grad=True)
loss = w[0] ** 2 + 2 * w[1] ** 2
loss.backward()
print(f'autograd grad: {w.grad.tolist()}')
# Expected: [2.0, 4.0]

</details>

---

## Exercise 4 - The zero_grad bug

**Task:** Train the same tiny network twice, once forgetting `opt.zero_grad()` and once calling it, and print both loss curves.

**Expected:** The buggy run explodes to inf; the fixed run decreases.

In [ ]:
# YOUR CODE HERE
torch.manual_seed(0)
x = torch.randn(64, 4)
y = torch.randn(64, 1)

def train(zero_grad):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    losses = []
    for step in range(20):
        # TODO: if `zero_grad` is True, clear grads with opt.zero_grad()
        #       then forward -> F.mse_loss -> backward -> opt.step(), and record the loss
        #       (guard against inf so the loop can finish)
        ...
    return losses

# TODO: run train(zero_grad=False) vs train(zero_grad=True) and print both curves

<details><summary>💡 Hint</summary>

Without `opt.zero_grad()`, PyTorch accumulates gradients across steps, so the effective step grows and the loss blows up.
</details>

**Criteria:** Buggy curve reaches inf; fixed curve decreases.

<details><summary>✅ Solution</summary>


In [ ]:
torch.manual_seed(0)
x = torch.randn(64, 4)
y = torch.randn(64, 1)

def train(zero_grad):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    losses = []
    for step in range(20):
        if zero_grad: opt.zero_grad()
        pred = model(x)
        l = F.mse_loss(pred, y)
        l.backward()
        opt.step()
        losses.append(l.item())
        if l.item() > 1e10:
            losses.extend([float('inf')] * (20 - step - 1))
            break
    return losses

buggy = train(zero_grad=False)
fixed = train(zero_grad=True)
print('step  buggy        fixed')
for i, (b, f) in enumerate(zip(buggy, fixed)):
    print(f'{i:4d}  {b:.3e}  {f:.3e}')

</details>

---

## Exercise 5 - Cross-entropy vs MSE convergence

**Task:** Train a classifier with cross-entropy on logits vs MSE on softmax probabilities and compare how fast each loss drops.

**Expected:** Cross-entropy reaches a lower loss faster than MSE.

In [ ]:
# YOUR CODE HERE
torch.manual_seed(0)
N, D, C = 256, 8, 4
X = torch.randn(N, D)
W_true = torch.randn(D, C)
y = (X @ W_true).argmax(dim=1)
y_onehot = F.one_hot(y, C).float()

def train(loss_fn_kind):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(D, 16), nn.ReLU(), nn.Linear(16, C))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    losses = []
    for step in range(200):
        # TODO: forward pass, then pick the loss by loss_fn_kind:
        #   'ce'  -> F.cross_entropy(logits, y)
        #   'mse' -> F.mse_loss(logits.softmax(dim=1), y_onehot)
        # backward + step, record loss
        ...
    return losses

# TODO: train('ce') vs train('mse'); print final and halfway (step 100) losses

<details><summary>💡 Hint</summary>

Cross-entropy acts on logits directly and has a steeper early gradient; MSE-on-softmax flattens near confident predictions and learns slower.
</details>

**Criteria:** Both curves printed; CE reaches a lower loss faster than MSE.

<details><summary>✅ Solution</summary>


In [ ]:
torch.manual_seed(0)
N, D, C = 256, 8, 4
X = torch.randn(N, D)
W_true = torch.randn(D, C)
y = (X @ W_true).argmax(dim=1)
y_onehot = F.one_hot(y, C).float()

def train(loss_fn_kind):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(D, 16), nn.ReLU(), nn.Linear(16, C))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    losses = []
    for step in range(200):
        opt.zero_grad()
        logits = model(X)
        if loss_fn_kind == 'ce':
            l = F.cross_entropy(logits, y)
        else:
            l = F.mse_loss(logits.softmax(dim=1), y_onehot)
        l.backward()
        opt.step()
        losses.append(l.item())
    return losses

ce_curve = train('ce')
mse_curve = train('mse')
print(f'CE  final: {ce_curve[-1]:.4f}, halfway: {ce_curve[100]:.4f}')
print(f'MSE final: {mse_curve[-1]:.4f}, halfway: {mse_curve[100]:.4f}')

</details>

---

## Exercise 6 - Gradient clipping stability

**Task:** Train through batches that occasionally spike in magnitude, with and without gradient-norm clipping, and compare the worst-case loss.

**Expected:** Max loss without clipping is far larger than with clipping.

In [ ]:
# YOUR CODE HERE
def train(use_clip):
    torch.manual_seed(0)
    model = nn.Linear(4, 1)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    losses = []
    for step in range(50):
        opt.zero_grad()
        x = torch.randn(32, 4)
        if step % 10 == 9: x = x * 50.0   # inject a magnitude spike every 10 steps
        y = x.sum(dim=1, keepdim=True) + torch.randn(32, 1) * 0.1
        pred = model(x)
        l = F.mse_loss(pred, y)
        l.backward()
        # TODO: if use_clip, clip the gradient norm to max_norm=1.0 BEFORE opt.step()
        ...
        opt.step()
        losses.append(l.item())
    return losses

# TODO: compare max(loss) with and without clipping

<details><summary>💡 Hint</summary>

`torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` rescales the gradient after backward and before step, capping the damage from spike batches.
</details>

**Criteria:** Max loss without clipping is far larger than with clipping.

<details><summary>✅ Solution</summary>


In [ ]:
def train(use_clip):
    torch.manual_seed(0)
    model = nn.Linear(4, 1)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    losses = []
    for step in range(50):
        opt.zero_grad()
        x = torch.randn(32, 4)
        if step % 10 == 9: x = x * 50.0
        y = x.sum(dim=1, keepdim=True) + torch.randn(32, 1) * 0.1
        pred = model(x)
        l = F.mse_loss(pred, y)
        l.backward()
        if use_clip:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        losses.append(l.item())
    return losses

noclip = train(False)
withclip = train(True)
print(f'without clip: max loss = {max(noclip):.2e}')
print(f'with clip:    max loss = {max(withclip):.2e}')

</details>

---

## Exercise 7 - Sigmoid vanishing gradients

**Task:** Stack 10 layers and read off the per-layer gradient norms for Sigmoid vs ReLU activations.

**Expected:** Sigmoid's early-layer gradients are orders of magnitude smaller than ReLU's.

In [ ]:
# YOUR CODE HERE
def build(activation):
    layers = []
    in_dim = 4
    for i in range(10):
        layers.append(nn.Linear(in_dim, 4))
        layers.append(activation())
        in_dim = 4
    layers.append(nn.Linear(4, 1))
    return nn.Sequential(*layers)

def grad_norms(model, x, y):
    # TODO: forward -> F.mse_loss -> backward, then return
    #       m.weight.grad.norm() for every nn.Linear layer (input -> output order)
    ...

torch.manual_seed(0)
x = torch.randn(32, 4)
y = torch.randn(32, 1)
# TODO: print grad norms for build(nn.Sigmoid) and build(nn.ReLU); compare the early layers

<details><summary>💡 Hint</summary>

Sigmoid saturates, so its derivative is near zero; stacked 10 deep the gradient shrinks by orders of magnitude toward the input. ReLU keeps gradients healthier.
</details>

**Criteria:** Sigmoid early-layer grad norms are orders of magnitude smaller than ReLU's.

<details><summary>✅ Solution</summary>


In [ ]:
def build(activation):
    layers = []
    in_dim = 4
    for i in range(10):
        layers.append(nn.Linear(in_dim, 4))
        layers.append(activation())
        in_dim = 4
    layers.append(nn.Linear(4, 1))
    return nn.Sequential(*layers)

def grad_norms(model, x, y):
    pred = model(x)
    l = F.mse_loss(pred, y)
    l.backward()
    return [m.weight.grad.norm().item() for m in model if isinstance(m, nn.Linear)]

torch.manual_seed(0)
x = torch.randn(32, 4)
y = torch.randn(32, 1)
print('Sigmoid grad norms (input -> output):')
print([f'{v:.2e}' for v in grad_norms(build(nn.Sigmoid), x, y)])
print('ReLU grad norms (input -> output):')
print([f'{v:.2e}' for v in grad_norms(build(nn.ReLU), x, y)])

</details>

---

## Exercise 8 - Cosine LR with warmup

**Task:** Implement a learning-rate schedule with linear warmup followed by cosine decay, then sample and plot it.

**Expected:** LR ramps to peak by the end of warmup and decays smoothly to ~10% of peak.

In [ ]:
# YOUR CODE HERE
import math

def get_lr(step, total_steps, peak_lr=1e-3, warmup_frac=0.02, min_lr_frac=0.1):
    warmup_steps = int(total_steps * warmup_frac)
    # TODO: linear warmup while step < warmup_steps  ->  peak_lr * step / warmup_steps
    #       afterwards cosine-decay from peak_lr down to peak_lr * min_lr_frac
    ...

total = 1000
# TODO: build the schedule, print LR at steps 0, 20, 500, 999, and plot it

<details><summary>💡 Hint</summary>

Warmup: `peak_lr * step / warmup_steps`. Cosine: `0.5 * (1 + cos(pi * progress))` scaled between the min and peak LR.
</details>

**Criteria:** LR rises to peak by step 20 and decays to ~10% of peak by step 999.

<details><summary>✅ Solution</summary>


In [ ]:
import math

def get_lr(step, total_steps, peak_lr=1e-3, warmup_frac=0.02, min_lr_frac=0.1):
    warmup_steps = int(total_steps * warmup_frac)
    if step < warmup_steps:
        return peak_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cos = 0.5 * (1 + math.cos(math.pi * progress))
    return peak_lr * (min_lr_frac + (1 - min_lr_frac) * cos)

total = 1000
schedule = [get_lr(s, total) for s in range(total)]
print(f'step 0:    LR = {schedule[0]:.2e}')
print(f'step 20:   LR = {schedule[20]:.2e} (end of warmup)')
print(f'step 500:  LR = {schedule[500]:.2e} (midway)')
print(f'step 999:  LR = {schedule[999]:.2e} (10% of peak)')

plt.plot(schedule)
plt.xlabel('step'); plt.ylabel('LR'); plt.title('Cosine schedule with linear warmup')
plt.show()

</details>

---

## Wrap-up

Exercise 4 and Exercise 7 are the two production-training failure modes you are most likely to encounter. Exercise 8 (cosine schedule) is the production standard.